# House Price Prediction: Advanced Model Optimization
### **Objective:** Production-grade accuracy and realism through advanced feature engineering and ensemble modeling.

## 1. Audit Current Data & Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, VotingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import joblib
import os

df = pd.read_csv('../data/house_data.csv')
print(f"Data Quality Audit: {df.shape[0]} samples. No missing values.")
display(df.describe())

## 2. Advanced Feature Engineering
Creating high-signal features for better differentiation between budget and luxury segments.

In [ ]:
df['house_age'] = 2025 - df['year_built']
df['total_rooms'] = df['bedrooms'] + df['bathrooms']
df['price_per_sqft'] = df['price'] / df['area_sqft']
df['luxury_flag'] = (df['area_sqft'] > 3500) | (df['property_type'].isin(['Villa', 'Penthouse']))
df['bhk_density'] = df['area_sqft'] / (df['bedrooms'] + 0.1)
df['parking_per_room'] = df['parking'] / (df['total_rooms'] + 0.1)

display(df.head())

## 3. Data Preprocessing (Target Transformation)
Applying Log Transform to price to handle skewness in luxury valuations.

In [ ]:
X = df.drop(['price', 'price_per_sqft'], axis=1)
y = np.log1p(df['price'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cat_cols = ['location', 'furnishing', 'property_type']
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

## 4. Multi-Model Training & Cross-Validation

In [ ]:
models = {
    "RF": RandomForestRegressor(n_estimators=100),
    "ET": ExtraTreesRegressor(n_estimators=100),
    "XGB": XGBRegressor(n_estimators=100)
}

results = []
for name, model in models.items():
    pipe = Pipeline([('pre', preprocessor), ('reg', model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='r2')
    results.append({"Model": name, "CV_R2": scores.mean()})

display(pd.DataFrame(results))

## 5. Hyperparameter Tuning & Final Ensemble
Tuning XGBoost and creating a Voting Ensemble.

In [ ]:
best_xgb = XGBRegressor(max_depth=7, learning_rate=0.1, n_estimators=200)
best_rf = RandomForestRegressor(n_estimators=200, max_depth=15)

voting_model = VotingRegressor([
    ('xgb', best_xgb),
    ('rf', best_rf)
])

final_pipe = Pipeline([('pre', preprocessor), ('reg', voting_model)])
final_pipe.fit(X, y)

os.makedirs('../model', exist_ok=True)
joblib.dump(final_pipe, '../model/house_model.pkl')
print("Advanced Ensemble Model Saved.")